In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
import os
os.chdir('/content/drive/My Drive/DL/final_project')
path = os.getcwd()
print('path: ' + path)

In [ ]:
import cv2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import warnings
import pyperclip
import math

**Code for Data Augmentation**

In [ ]:
def mirror_image(img, x_pixel):

    height, width, channels = img.shape
    mirrored_x_index = width - (x_pixel + 1)

    mirrored_img_array = np.fliplr(img)

    return mirrored_img_array, mirrored_x_index


def create_reflection_data():
  file_path = 'length_measurements.csv'
  original_df = pd.read_csv(file_path)
  new_idx = original_df.shape[0]
  df = original_df.dropna(subset=['pixel_location'])
  df[['pixel_x', 'pixel_y']] = df['pixel_location'].str.extract(r'\((\d+),\s*(\d+)\)').astype(int)

  df.index += 2

  df['images'] = df.apply(lambda row: f'{row["Filename"][:-4]}_number_{row.name}.jpg', axis=1)

  new_df = pd.DataFrame(columns=df.columns)

  for index, row in df.iterrows():
    image_path = f'data_images/{row["images"]}'
    img = cv2.imread(image_path)
    x_pixel = row['pixel_x']
    mirrored_image, mirrored_x = mirror_image(img, x_pixel)

    new_row = row.copy()
    new_row['pixel_location'] = f'({mirrored_x}, {row["pixel_y"]})'
    cv2.imwrite(f'data_images/{row["Filename"][:-4]}_number_{new_idx + 2}.jpg', mirrored_image)
    new_df.loc[new_idx] = new_row
    new_idx += 1

  new_df.drop(columns=['images', 'pixel_x', 'pixel_y'], inplace=True)
  updated_df = pd.concat([original_df, new_df], ignore_index=False)


  updated_df.to_csv('length_measurements_reflected.csv', index=True)







In [ ]:
create_reflection_data()

**Code for extracting distances of data points**

In [ ]:
warnings.filterwarnings("ignore")
plt.rcParams["figure.max_open_warning"] = 0


def onclick(event):
    if event.xdata is not None and event.ydata is not None:
        x, y = int(event.xdata), int(event.ydata)
        print(f"\nClicked position: x={x}, y={y}")
        if img is not None:
            pixel_value = f"({x},{y})"
            pyperclip.copy(pixel_value)
            print(f"Copied to clipboard: {pixel_value}")

# set the of require video to extract coordinates
video_name = 'lGP063317'
index = 4175

img = mpimg.imread(
    f'data_images/{video_name}_number_{index}.jpg'
)
print(f"Image dimensions: {img.shape[1]}x{img.shape[0]} pixels")

fig, ax = plt.subplots()
ax.imshow(img)
fig.canvas.mpl_connect("button_press_event", onclick)
plt.show()


In [ ]:
def extract_frame_at_time(video_path, time_float, output_image_path="data_images/extracted_frame.jpg"):

    # Open video
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f"Unable to open video file: {video_path}")

    # Convert time_float to total seconds
    minutes = int(time_float)
    seconds = (time_float - minutes) * 60
    total_seconds = minutes * 60 + seconds

    # Set video to the correct position in milliseconds
    cap.set(cv2.CAP_PROP_POS_MSEC, total_seconds * 1000)

    # Read the frame at that position
    success, frame = cap.read()

    if not success:
        raise ValueError(f"Could not read frame at {time_float:.5f} minutes ({total_seconds:.2f} seconds)")

    # Ensure output directory exists
    os.makedirs(os.path.dirname(output_image_path), exist_ok=True)

    # Save the frame as an image
    cv2.imwrite(output_image_path, frame)
    print(f"Frame at {time_float:.5f} minutes saved as {output_image_path}")

    cap.release()


def get_video_length_cv(video_path):
    """Returns the length of the video in milliseconds using OpenCV."""
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError("Error: Could not open video file.")

    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT)

    duration_ms = round((frame_count / fps) * 1000) if fps > 0 else 0
    duration_min = duration_ms / 60000
    cap.release()
    return duration_min



file_path = 'Metadata.xlsx - length_measurements.csv'
df = pd.read_csv(file_path)

columns_to_keep = ['lGP014922.MP4', 'Time (mins)']
df = df[columns_to_keep]

videos_to_keep = [
    'lGP014912.MP4', 'lGP064912.MP4', 'lGP084912.MP4'
]

df = df[df['lGP014922.MP4'].isin(videos_to_keep)]

cycle = get_video_length_cv(f'videos/{videos_to_keep[1]}')

def calculate_absolute_time(time_mins, cycle_length):
    while time_mins >= cycle_length:
        time_mins -= cycle_length
    return time_mins


# Apply the function to create the new column
df['absolute_time'] = df['Time (mins)'].apply(calculate_absolute_time, args=(cycle,))

df['minutes'] = df['absolute_time'].apply(lambda x: math.floor(x))

df['second'] = df['absolute_time'].apply(lambda x: int((x - math.floor(x)) * 60))


df.rename(columns={'lGP014922.MP4': 'VideoName'}, inplace=True)

df.to_csv('output_file.csv', index=False)


video_directory = 'videos'

for index, row in df.iterrows():
    video_filename = row['VideoName']
    time = row['absolute_time']
    video_path = os.path.join(video_directory, video_filename)

    output_image_path = f"data_images/{video_filename[:-4]}_number_{index + 2}.jpg"

    try:
        extract_frame_at_time(video_path, time,  output_image_path)
    except Exception as e:
        print(f"Failed to extract frame from {video_filename}: {e}")